# Resume-to-Job-Description Matcher — Siamese Transformer (built from scratch)

Loads the `resume_jd_pairs.csv` built in `ResumeJD_Logistic_Regression.ipynb` and trains a
**Siamese Transformer encoder**, implemented from scratch (custom positional encoding +
custom multi-head self-attention + feed-forward block — no pretrained weights, no
Hugging Face pipeline), mirroring the from-scratch transformer classifier approach used
elsewhere. Both resume and JD are encoded through the *same* shared transformer branch,
then merged for the final match/no-match classifier head — same overall shape as the
SimpleRNN/LSTM/GRU Siamese notebooks, with the RNN branch swapped for a transformer
encoder.

In [ ]:
import re
import time
import pickle
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    confusion_matrix, ConfusionMatrixDisplay, classification_report
)

import tensorflow as tf
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.models import Model
from tensorflow.keras.layers import (
    Input, Embedding, Layer, Dense, Dropout, LayerNormalization,
    GlobalAveragePooling1D, Concatenate, Lambda
)
from tensorflow.keras.callbacks import EarlyStopping

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)
tf.random.set_seed(RANDOM_STATE)

print("TensorFlow version:", tf.__version__)


## 1. Load the pairs dataset

In [ ]:
pairs = pd.read_csv("resume_jd_pairs.csv")
pairs.drop_duplicates(inplace=True)
pairs.dropna(inplace=True)
print("Shape:", pairs.shape)
pairs.head(3)


## 2. Train/test split

In [ ]:
X_train_text, X_test_text, y_train, y_test = train_test_split(
    pairs[["resume", "jd"]], pairs["label"],
    test_size=0.2, random_state=RANDOM_STATE, stratify=pairs["label"]
)


## 3. Tokenize + pad (shared tokenizer, same as the RNN notebooks)

In [ ]:
MAX_LEN_RESUME = 300
MAX_LEN_JD = 150
VOCAB_SIZE = 15000

tokenizer = Tokenizer(num_words=VOCAB_SIZE, oov_token="<OOV>")
tokenizer.fit_on_texts(pd.concat([X_train_text["resume"], X_train_text["jd"]]))

def encode(df):
    resume_seq = tokenizer.texts_to_sequences(df["resume"])
    jd_seq = tokenizer.texts_to_sequences(df["jd"])
    resume_pad = pad_sequences(resume_seq, maxlen=MAX_LEN_RESUME, padding="post", truncating="post")
    jd_pad = pad_sequences(jd_seq, maxlen=MAX_LEN_JD, padding="post", truncating="post")
    return resume_pad, jd_pad

X_train_resume, X_train_jd = encode(X_train_text)
X_test_resume, X_test_jd = encode(X_test_text)

y_train_arr = y_train.values
y_test_arr = y_test.values


## 4. Transformer building blocks (from scratch)

### 4a. Positional encoding
Since a transformer has no built-in notion of word order, we add the standard sinusoidal
positional encoding to the token embeddings.

In [ ]:
def get_positional_encoding(max_len, d_model):
    positions = np.arange(max_len)[:, np.newaxis]
    dims = np.arange(d_model)[np.newaxis, :]
    angle_rates = 1 / np.power(10000, (2 * (dims // 2)) / np.float32(d_model))
    angle_rads = positions * angle_rates

    angle_rads[:, 0::2] = np.sin(angle_rads[:, 0::2])
    angle_rads[:, 1::2] = np.cos(angle_rads[:, 1::2])

    return tf.cast(angle_rads[np.newaxis, ...], dtype=tf.float32)


class PositionalEmbedding(Layer):
    def __init__(self, max_len, vocab_size, d_model, **kwargs):
        super().__init__(**kwargs)
        self.token_embedding = Embedding(vocab_size, d_model, mask_zero=True)
        self.pos_encoding = get_positional_encoding(max_len, d_model)
        self.d_model = d_model

    def call(self, x):
        seq_len = tf.shape(x)[1]
        embedded = self.token_embedding(x)
        embedded *= tf.math.sqrt(tf.cast(self.d_model, tf.float32))
        return embedded + self.pos_encoding[:, :seq_len, :]

    def compute_mask(self, x, mask=None):
        return self.token_embedding.compute_mask(x)


### 4b. Multi-head self-attention + feed-forward encoder block

Built from the primitive `Dense` layers rather than a single pre-packaged attention layer,
so every projection (Q, K, V, output) is explicit.

In [ ]:
class MultiHeadSelfAttention(Layer):
    def __init__(self, d_model, num_heads, **kwargs):
        super().__init__(**kwargs)
        assert d_model % num_heads == 0
        self.num_heads = num_heads
        self.d_model = d_model
        self.depth = d_model // num_heads

        self.wq = Dense(d_model)
        self.wk = Dense(d_model)
        self.wv = Dense(d_model)
        self.dense = Dense(d_model)

    def split_heads(self, x, batch_size):
        x = tf.reshape(x, (batch_size, -1, self.num_heads, self.depth))
        return tf.transpose(x, perm=[0, 2, 1, 3])

    def call(self, x, mask=None):
        batch_size = tf.shape(x)[0]

        q = self.split_heads(self.wq(x), batch_size)
        k = self.split_heads(self.wk(x), batch_size)
        v = self.split_heads(self.wv(x), batch_size)

        scores = tf.matmul(q, k, transpose_b=True) / tf.math.sqrt(tf.cast(self.depth, tf.float32))

        if mask is not None:
            mask = tf.cast(mask[:, tf.newaxis, tf.newaxis, :], tf.float32)
            scores += (1.0 - mask) * -1e9

        weights = tf.nn.softmax(scores, axis=-1)
        attention = tf.matmul(weights, v)

        attention = tf.transpose(attention, perm=[0, 2, 1, 3])
        attention = tf.reshape(attention, (batch_size, -1, self.d_model))
        return self.dense(attention)


class TransformerEncoderBlock(Layer):
    def __init__(self, d_model, num_heads, ff_dim, dropout_rate=0.1, **kwargs):
        super().__init__(**kwargs)
        self.attention = MultiHeadSelfAttention(d_model, num_heads)
        self.ffn = tf.keras.Sequential([
            Dense(ff_dim, activation="relu"),
            Dense(d_model),
        ])
        self.norm1 = LayerNormalization(epsilon=1e-6)
        self.norm2 = LayerNormalization(epsilon=1e-6)
        self.drop1 = Dropout(dropout_rate)
        self.drop2 = Dropout(dropout_rate)

    def call(self, x, mask=None, training=False):
        attn_out = self.attention(x, mask=mask)
        x = self.norm1(x + self.drop1(attn_out, training=training))

        ffn_out = self.ffn(x)
        return self.norm2(x + self.drop2(ffn_out, training=training))


## 5. Build the Siamese Transformer model

Both branches share the **same** positional-embedding layer and the **same** transformer
encoder block (true weight sharing). Each branch is pooled to a single fixed-size vector
(`GlobalAveragePooling1D`), then combined with concatenation + absolute difference, same
merge strategy as the RNN notebooks, before the classifier head.

In [ ]:
D_MODEL = 64
NUM_HEADS = 4
FF_DIM = 128

# Shared layers (weight-tied across both branches)
shared_pos_embedding_resume = PositionalEmbedding(MAX_LEN_RESUME, VOCAB_SIZE, D_MODEL)
shared_pos_embedding_jd = PositionalEmbedding(MAX_LEN_JD, VOCAB_SIZE, D_MODEL)
shared_encoder = TransformerEncoderBlock(D_MODEL, NUM_HEADS, FF_DIM)
pooling = GlobalAveragePooling1D()

resume_input = Input(shape=(MAX_LEN_RESUME,), name="resume_input")
jd_input = Input(shape=(MAX_LEN_JD,), name="jd_input")

resume_mask = tf.cast(tf.not_equal(resume_input, 0), tf.float32)
jd_mask = tf.cast(tf.not_equal(jd_input, 0), tf.float32)

resume_encoded = shared_encoder(shared_pos_embedding_resume(resume_input), mask=resume_mask)
jd_encoded = shared_encoder(shared_pos_embedding_jd(jd_input), mask=jd_mask)

resume_vec = pooling(resume_encoded)
jd_vec = pooling(jd_encoded)

abs_diff = Lambda(lambda x: tf.abs(x[0] - x[1]))([resume_vec, jd_vec])
merged = Concatenate()([resume_vec, jd_vec, abs_diff])

x = Dense(64, activation="relu")(merged)
x = Dropout(0.3)(x)
x = Dense(32, activation="relu")(x)
output = Dense(1, activation="sigmoid")(x)

transformer_model = Model(inputs=[resume_input, jd_input], outputs=output)

transformer_model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=0.0005),
    loss="binary_crossentropy",
    metrics=["accuracy"]
)

transformer_model.summary()


## 6. Train

In [ ]:
early_stop = EarlyStopping(
    monitor="val_loss",
    patience=4,
    restore_best_weights=True
)

history_transformer = transformer_model.fit(
    [X_train_resume, X_train_jd],
    y_train_arr,
    validation_split=0.1,
    epochs=20,
    batch_size=128,
    callbacks=[early_stop]
)


## 7. Evaluate on the test set

In [ ]:
y_pred_transformer = (transformer_model.predict([X_test_resume, X_test_jd]) > 0.5).astype(int)

print(classification_report(y_test_arr, y_pred_transformer))


In [ ]:
accuracy_score(y_test_arr, y_pred_transformer)


In [ ]:
cm = confusion_matrix(y_test_arr, y_pred_transformer)
ConfusionMatrixDisplay(cm, display_labels=["No Match", "Match"]).plot()
plt.title("Siamese Transformer — Confusion Matrix")
plt.show()


In [ ]:
plt.plot(history_transformer.history["accuracy"], label="train acc")
plt.plot(history_transformer.history["val_accuracy"], label="val acc")
plt.xlabel("Epoch")
plt.ylabel("Accuracy")
plt.legend()
plt.title("Siamese Transformer — Training curve")
plt.show()


## 8. Compare against the RNN variants and Logistic Regression baseline

Collect accuracy/F1 from this notebook alongside `ResumeJD_Logistic_Regression.ipynb`,
`ResumeJD_SimpleRNN.ipynb`, `ResumeJD_LSTM.ipynb`, and `ResumeJD_GRU.ipynb` into a single
comparison table for your README/resume writeup. If the Transformer outperforms the GRU,
swap `app.py` to load this model instead (it will need the two-branch predict call shown
above, plus this notebook's custom layer classes available at load time).

In [ ]:
with open("resume_jd_transformer_tokenizer.pkl", "wb") as f:
    pickle.dump(tokenizer, f)

transformer_model.save("resume_jd_transformer_model.keras")


In [ ]:
from google.colab import files

files.download("resume_jd_transformer_model.keras")
files.download("resume_jd_transformer_tokenizer.pkl")
